# ECS659U - Coursework


* The **goal** of the CW is similar to that of Week 2's Lab: fitting a curve to data, also known as **curve fitting**.
* This has applications in many different disciplines that make use of AI: FinTech, Physics Modelling, or even Sports.
* For example, we might be interested in learning the evolution (over time) of the price of a specific product in different countries. This can depend on several factors: the product itself, the country, the initial value of the product's price, etc.
* As usual, we are interested in learning a model that finds these relationships *from the data*.


## Learning a set of functions

* The main difference with Week 2's Lab that we will train a network that does not learn a single function but a set of functions.
* You are provivided with a training set ("train_data.csv") containing 30,000 functions. Each function has $100$ points (N_x = 100) on the x-axis from a $[-1, 1]$ distribution.
* Because we are dealing with a family of functions and not just a single function, our model must be able to perform two tasks: *Function Selection* and *Regression*.
* Function selection means that given some *additional* input (to be defined below) the model somehow must choose which function from the set of functions it needs to model.
* Once the correct function is picked then the model must perform regression i.e. learn the input-output relationship $y=f_a(x)$ where $f_a$ is the selected function.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/CW

Mounted at /content/drive
/content/drive/MyDrive/CW


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

y_train = np.loadtxt('train_data.csv', delimiter=',')
N_train, Nx = y_train.shape
x_train = np.tile(np.linspace(-1, 1, Nx), (N_train, 1))
x_tensor = torch.from_numpy(x_train).float()
y_tensor = torch.from_numpy(y_train).float()
train_loader = DataLoader(TensorDataset(x_tensor, y_tensor),batch_size=128,shuffle=True)

y_test = torch.from_numpy(np.loadtxt('test_data.csv', delimiter=',')).float()
x_test = torch.linspace(-1, 1, 100).unsqueeze(0).repeat(300, 1)

# Load the context bundle
test_contexts = torch.load('test_observations.pt')


## The Learning Objective


* For both *training and testing*, a random subset of $N$ pairs $(x_o, y_o)$ (which is called observed data) is provided. This acts as prior information to help the model identify which specific function is looking at.

* The model will take the observed pairs $(x_o, y_o)$ and target values $x_t$ and will produce the estimated values $\hat{y}_t$. The ultimate goal is to estimate the function corresponding to the curve that observed pairs $(x_o, y_o)$ belong to and then estimate the output value $\hat{y}_t$ corresponding to $x_t$. During training we have access to the ground-truth values $y_t$, and thus we can compute a loss between the model's predictions $\hat{y}_t$ and the ground-truth values $y_t$.  


## The Model Architecture

* Our model consists of 2 MLPs which must be jointly trained. Specifically, you should implement the following architecture:
* *Encoder*: The first MLP is called the Encoder. It takes as input the random set of $N$ observed pairs $(x_o, y_o)$  and maps each of them (through a series of hidden layers) to a feature vector of dimension $h_{dim}$. A final layer produces the latent feature representation $r_o$ of dimension $r_{dim}$.
* A total observed feature is produced by summing or averaging over all features (e.g., $r_O=\frac{1}{N}\sum_o r_o$).
* *Decoder*: The second MLP is called the Decoder. It takes as input the $r_O$ and each input data $x_t$. The Decoder (through a series of hidden layers) maps the pair $(r_O, x_t)$ to some feature vector of dimension $h_{dim}$. A final layer will produce the model's prediction $\hat{y}_t$ (corresponding to the input $x_t$).
* Hint! Try different choices for the number of hidden layers in the MLPs and for the hidden dimension $h_{dim}$.


## Tasks

* You have to implement the following:
    1. Create the model based on the provided architecture description. $r_{dim}$ should be a configurable hyperparameter. (20%)
    2. Create the optimizer and the loss function. Write the training script that will train the model and print the training loss. (20%)
    3. Train three versions of the model, called model2, model4, model8, by only alternating the dimension of the latent representation $r_{dim}$ to be 2, 4, and 8, respectively. Using the provided test_data.csv and the observation pairs provided in test_observations.pt, calculate the average error for each model. Provide results for all these calculations in your report. (30%)
    4.  Given you results, which latent representation ( $r_{dim}$ ) yielded the best results and which latent representation ($r_{dim}$) offered the  most significant improvement? How many independent variables were used to generate the training and test data? Provide explanation. (20%)
    5. For $r_{dim}$ which offered the most improvement, how capable your model was in terms of disentagling the independent variables? Hint: check the correlations between the dimensions of the latent representation. Provide explanation. (10%)

# Notes
**xi yi** Training examples, input and output of a function

**Latent Representation**
Shows information about function, like slope, intercept

**Latent Vector Size**
r_dim - if its 2 then the vector has 2 numbers in it - [0.2, 0.3]

**Encoder output**
example take mean(all ri), output single r,

decoder will use this to predict new y value. ri = encoder(xi, yi)

In [12]:
# Imports
from torch import nn

In [14]:
# Helper function for weights
def init_weights(m):
  if isinstance(m, nn.Linear):
    torch.nn.init.normal_(m.weight, std=0.01)
    torch.nn.init.zeros_(m.bias)

The below MLP class will make every mlp have 2 hidden layers

Each hidden layer will have 64 neurons

After each hidden layer, the relu funciton will be applied to the output

In [15]:
# MLP Classx

class Net(torch.nn.Module):
  def __init__(self, num_inputs, num_outputs):
    super(Net, self).__init__()
    self.num_inputs = num_inputs
    self.num_outputs = num_outputs
    num_hidden = 64

    self.Linear1 = nn.Linear(num_inputs, num_hidden)
    self.Linear2 = nn.Linear(num_hidden, num_hidden)
    self.Linear3 = nn.Linear(num_hidden, num_outputs)

  def forward(self, x):
    out = torch.relu(self.Linear1(x))
    out = torch.relu(self.Linear2(out))
    out = self.Linear3(out)
    return out



# Implement the encoder and decoder MLPs
*   Each should have configurable number of input and output dimensions

**Encoder**
Transforms the input data to a latent output vector for the decoder to use
1.   Takes context set as input: xi yi pairs
2.   Maps each pair to a latent representation vector ri
3.   Take average of representations to show single context representation


**Decoder**
Uses latent representation of the data, and the new input that we want a prediction for

In [16]:
class CombinedModel(nn.Module):
  def __init__(self, x_dim, y_dim, r_dim):
    super().__init__()

    # Encoder
    self.encoder = Net(
        num_inputs = x_dim + y_dim,
        num_outputs = r_dim
    )

    # Decoder
    self.decoder = Net(
        num_inputs = x_dim + r_dim,
        num_outputs = y_dim
    )

  def forward(self, x_context, y_context, x_target):
    # Concatenate the columns of x and y, so they have the shape
    # [x1, y1], [x2, y2] ect
    # Number of rows stay the same, the columns were just added together
    xy = torch.cat([x_context, y_context], dim=-1)

    # We calculate rows of r_i, and take the average
    # We want one value per column for the aggregated r vector
    # So average accross dimension 0
    # Gives us the representations from all context points
    r_i = self.encoder(xy)
    r = torch.mean(r_i, dim=0)

    # Copies of r need to be concatenated to elements of x_context
    # in every row, so r needs to be made into 2d
    # Each target point gets same r vector
    r = r.unsqueeze(0)
    r = r.repeat(x_target.shape[0], 1)

    # Put the x values together with the r values,
    # x defines where we want to evaluate the function
    # r values are describing the function
    decoder_input = torch.cat([x_target, r], dim=-1)
    y_pred = self.decoder(decoder_input)

    return y_pred

# Setting up the training process

**Optimiser**
Set up optimiser, using SGD (Stoachastic Gradient Descent) with learning rate and weight decay

**Loss Function**
Measure difference between true outputs and predicted outputs. Use MSE or Cross Entropy Loss

**Training Loop**
*   Each iteration, sample a function from training set.
*   Split observations
*   Context set: observed points used to infer function
*   Target set: points the model must predict
*   Pass (x, y) through encoder
*   Produce ri, and compute the aggregated mean
*   Decode predictions, combine r with target prediction xt
*   Compute loss with the predicted y hat t with true yt
*   Backpropagation, compute gradients and optimiser update network weights



In [17]:
def train_model(model, train_loader, epochs):
  lr = 0.1
  wd = 0
  optimiser = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
  loss_f = nn.MSELoss()

  model.train()

  for epoch in range(epochs):

    epoch_loss = 0

    for x_batch, y_batch in train_loader:

      # Ensure the points are treated as 1D input/output
      x_batch = x_batch.unsqueeze(-1)
      y_batch = y_batch.unsqueeze(-1)

      # One batch contains multiple functions
      # Shape gives the number of functions in the batch
      # Loop through the functinos to train the model
      for i in range(x_batch.shape[0]):

        x = x_batch[i]
        y = y_batch[i]

        # Use first n as the context points
        n_context = 10
        x_c = x[:n_context]
        y_c = y[:n_context]
        # Use all points as targets


        # Ensure the gradients from previous step are cleared
        # Each batch needs it's own gradients to adjust the model afterwards
        optimiser.zero_grad()

        # model predicts from context variables
        # then it uses the x targets to make prediction
        # find loss between predicted y and target y
        # The targets are just the variables themselves
        y_pred = model(x_c, y_c, x)
        loss = loss_f(y_pred, y)

        loss.backward()
        optimiser.step()

        epoch_loss += loss.item()

    print("Epoch: ", epoch, " Loss: ", epoch_loss)

Initialise model, train,

In [ ]:
model = CombinedModel(x_dim=1, y_dim=1, r_dim=64)
train_model(model, train_loader, epochs=100)


Epoch:  0  Loss:  12913.961993015604
Epoch:  1  Loss:  12856.178654909978
Epoch:  2  Loss:  12835.352035980788
Epoch:  3  Loss:  12839.315507105319
Epoch:  4  Loss:  12881.355996126775
Epoch:  5  Loss:  12917.638341954444
Epoch:  6  Loss:  12832.164728797157
Epoch:  7  Loss:  12761.584201789927
Epoch:  8  Loss:  1487865.3908026968
Epoch:  9  Loss:  18220.957422859035
Epoch:  10  Loss:  18180.09822588018
Epoch:  11  Loss:  18158.8294638698
Epoch:  12  Loss:  18184.19511052873
Epoch:  13  Loss:  18229.833114398178
Epoch:  14  Loss:  18206.498553044396
Epoch:  15  Loss:  18168.2248493745
Epoch:  16  Loss:  18235.448195956997
Epoch:  17  Loss:  18173.195453069347
Epoch:  18  Loss:  18198.37563456234
Epoch:  19  Loss:  18151.657000735053
Epoch:  20  Loss:  18217.902344916947
Epoch:  21  Loss:  18156.675994889345
Epoch:  22  Loss:  18209.568209193123
Epoch:  23  Loss:  18199.122330520302
Epoch:  24  Loss:  18148.927190868417
Epoch:  25  Loss:  18163.418829284434
Epoch:  26  Loss:  18167.9671